# Lab 1: your first model call

Send one message, add a system prompt, hold a conversation and read every field of the
reply. This notebook follows [Lab 1 in the book](https://handbook.aggarwal.dev/lab-first-call.html): read each
step there, then run it here. The code is in `labs/01-first-call/first_call.py`.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: send one message

The same command you would type in a terminal. On the fake backend the reply repeats what
it received, which proves the message travelled.

In [ ]:
!python labs/01-first-call/first_call.py "Say hello in five words"

## Step 2: read the reply

Call the lab's `ask()` from Python and look at the fields of the `Reply` it returns. On the
fake backend, use the lab's own stand-in model, as the script does.

In [ ]:
sys.path.insert(0, str(root / "labs" / "01-first-call"))
import first_call
from common import llm

if llm.backend_name() == "fake":
    llm.fake.responder = first_call.fake_model

reply = first_call.ask("What is a token? Answer in one sentence.")
print("text:       ", reply.text)
print("stop_reason:", reply.stop_reason)
print("tokens:     ", reply.input_tokens, "in,", reply.output_tokens, "out")
print("cost:       ", first_call.usd(reply.cost_usd))
print("on claude-haiku-5-5:", first_call.usd(first_call.reference_cost(reply.input_tokens, reply.output_tokens)))

## Step 3: add a system prompt

`--show-request` prints what is sent before sending it.

In [ ]:
!python labs/01-first-call/first_call.py --show-request --system "Answer in one sentence." "What is a token?"

## Step 4: hold a conversation

A model remembers nothing between calls: the conversation is a list you keep and send
whole every time. Watch the input tokens climb from turn to turn.

In [ ]:
chat = first_call.Conversation(system="You are a friendly travel planner.")
_ = first_call.run_chat([
    "I am planning a three-day cycling trip.",
    "What should I pack?",
    "What did I say I was planning?",
], chat)

Print what the third call actually sent:

In [ ]:
for message in chat.messages:
    print(message["role"], "|", str(message["content"])[:80])

## Step 5: when max_tokens is too small

The reply stops at the ceiling, mid-word if need be, and `stop_reason` says so.

In [ ]:
!python labs/01-first-call/first_call.py --max-tokens 8 "Explain tokens in three paragraphs"

## Step 6: compare backends

Change `BACKEND` in the setup cell, run it again, then run the cells above. Compare the
words, the token counts, the latency and the cost.

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 01-first-call

## Experiments

Try the experiments on the lab page, for example a longer conversation or `effort="low"`
on Claude, and edit the cells above to run them. Then tick the lab's checklist in the
book: your progress is kept in your browser, not in this notebook.